In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\processed')

tfidf_unigram_matrix = load_npz(processed_dir / "tfidf_unigram_matrix.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
df = pd.read_csv(processed_dir / 'sample_clean.csv')

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert tfidf_unigram_matrix.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(df["id"])

print("Shape :", tfidf_unigram_matrix.shape)

Shape : (500, 2036)


In [3]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=6,
    metric="cosine"
)

labels = dbscan.fit_predict(tfidf_unigram_matrix)

# Les résultats restent associés aux identifiants exportés.
clusters = ids.copy()
clusters["cluster"] = labels

display(clusters["cluster"].value_counts().sort_index())

clusters_nb = clusters.loc[
    clusters["cluster"] != -1, "cluster"
].nunique()

print("Nombre de clusters :", clusters_nb)

cluster
-1     335
 0      28
 1      55
 2       6
 3       6
 4       8
 5       6
 6       7
 7      15
 8       6
 9       7
 10     15
 11      6
Name: count, dtype: int64

Nombre de clusters : 12


In [4]:
without_noise = labels != -1

X_eval = tfidf_unigram_matrix[without_noise]
evaluated_labels = labels[without_noise]

clusters_nb = len(np.unique(evaluated_labels))

score = silhouette_score(
    X_eval,
    evaluated_labels,
    metric="cosine"
)

print(f"Silhouette without_noise : {score:.3f}")
print(f"Evaluated products : {without_noise.sum()} / {len(labels)}")
print("Noise_pct :", (labels == -1).mean() * 100)

Silhouette without_noise : 0.469
Evaluated products : 165 / 500
Noise_pct : 67.0


In [5]:
projection_2d = TruncatedSVD(n_components=2, random_state=42)

In [6]:
projection_2d

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",2
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",42
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [7]:
coordinates = projection_2d.fit_transform(tfidf_unigram_matrix)
visualisation = pd.DataFrame({
    "SVD 1": coordinates[:, 0],
    "SVD 2": coordinates[:, 1],
    "id": ids["id"].to_numpy(),
    "cluster": [
        "Noise" if label == -1 else f"Cluster {label}"
        for label in labels
    ]
})

In [8]:
cores = np.zeros(len(labels), dtype=bool)
cores[dbscan.core_sample_indices_] = True

visualisation["status"] = np.where(
    labels == -1,
    "Noise",
    np.where(cores, "core", "border")
)

fig = px.scatter(
    visualisation,
    x="SVD 1",
    y="SVD 2",
    color="cluster",
    symbol="status",
    symbol_map={"Core": "circle", "Border": "diamond", "Noise": "x"},
    color_discrete_map={"Noise": "lightgray"},
    hover_data=["id", "status"],
    opacity=0.7,
    title="DBSCAN — clusters et points core",
    height=600,
    width=600
)

fig.show()

In [9]:
tsne = TSNE(
    n_components=2,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_tsne = tsne.fit_transform(tfidf_unigram_matrix.toarray())

visu_tsne = visualisation.copy()
visu_tsne["t-SNE 1"] = coord_tsne[:, 0]
visu_tsne["t-SNE 2"] = coord_tsne[:, 1]

fig = px.scatter(
    visu_tsne,
    x="t-SNE 1",
    y="t-SNE 2",
    color="cluster",
    symbol="status",
    symbol_map={"Core": "circle", "Border": "diamond", "Noise": "x"},
    color_discrete_map={"Noise": "lightgray"},
    hover_data=["id", "status"],
    opacity=0.7,
    title="DBSCAN — visualisation t-SNE, distance cosinus",
    height=600,
    width=600
)

fig.show()

In [10]:
tsne_3d = TSNE(
    n_components=3,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_3d = tsne_3d.fit_transform(tfidf_unigram_matrix.toarray())

visu_3d = visualisation.copy()
visu_3d[["t-SNE 1", "t-SNE 2", "t-SNE 3"]] = coord_3d

fig = px.scatter_3d(
    visu_3d,
    x="t-SNE 1",
    y="t-SNE 2",
    z="t-SNE 3",
    color="cluster",
    symbol="status",
    symbol_map={"Core": "circle", "Border": "diamond", "Noise": "x"},
    color_discrete_map={"Noise": "lightgray"},
    hover_data=["id", "status"],
    opacity=0.7,
    title="DBSCAN — projection t-SNE en 3D",
    height=600,
    width=600
)

fig.update_traces(marker_size=4)
fig.update_layout(scene_aspectmode="data")
fig.show()

Le cluster -1 est le cluster considéré comme du bruit par l'algorithme Dbscan. Nous allons tenter de réduire le nombre de produits dans ce cluster en ajustant les paramètres de l'algorithme.

In [11]:
vectorizer = joblib.load("../models/tfidf_unigram_vectorizer.joblib")
words = vectorizer.get_feature_names_out()

# Retrouver les descriptions dans l'ordre exact de la matrice
source = pd.read_csv(processed_dir / 'sample_clean.csv'
).set_index("id")

reading = ids.copy()
reading["cluster"] = labels
reading["text"] = source.loc[
    ids["id"], "description_text"
].to_numpy()

assert tfidf_unigram_matrix.shape == (len(reading), len(words))

In [12]:
cluster_ids = np.unique(labels[labels != -1]) # Parcours tous les clusters sauf le bruit (-1)

for cluster_id in cluster_ids:
    mask = labels == cluster_id
    assert cluster_id != -1 and mask.any()

    X_cluster = tfidf_unigram_matrix[mask]

    # Poids TF-IDF moyen de chaque mot dans ce groupe
    weight = np.asarray(X_cluster.mean(axis=0)).ravel()
    top = np.argsort(weight)[::-1][:12]
    top = top[weight[top] > 0]

    words_cluster = pd.DataFrame({
        "word": words[top],
        "mean weight TF-IDF": weight[top]
    })

    fig = px.bar(
        words_cluster,
        x="mean weight TF-IDF",
        y="word",
        orientation="h",
        title=f"Cluster {cluster_id} — majority words",
        height=400,
        width=600
    )
    fig.update_yaxes(categoryorder="total ascending")
    fig.show()

    # Descriptions les plus proches du centre moyen en distance cosinus
    center = np.asarray(X_cluster.mean(axis=0))
    similarities = cosine_similarity(X_cluster, center).ravel()

    positions = np.flatnonzero(mask)
    selection = positions[np.argsort(similarities)[::-1][:3]]

    print(f"Cluster {cluster_id} : {len(positions)} products")

    with pd.option_context("display.max_colwidth", None):
        display(reading.iloc[selection][["id", "text"]])

Cluster 0 : 28 products


,id,text
496,497,"cap 2 crew - this crew takes the edge off fickle weather. in clearing conditions, it has the fastest dry time of all our performance baselayer fabrics, with an open-knit construction that breathes and wicks to keep you dry and cool. when the weather window shuts, you can wear it as insulation beneath layers. with a self-fabric neck that s soft against the skin, and raglan sleeves and flat-ribbed underarm panels that merge out of the way of shoulder straps to eliminate chafe. made from 4-oz 100 % polyester 54% recycled with gladiodor natural odor control for the garment. recyclable through the common threads recycling program."
21,22,"cap 2 t-shirt - this one revels in the sweat jamborees of hill climbs, ridge runs and desert singletrack under a resolute sun. made from the fastest-drying of our performance baselayer fabrics, it has an open knit that invites airflow and provides excellent wicking and exceptional breathability. when worn beneath layers, it s also insulative enough for chilly days. the self-binding neck is pliable and soft raglan sleeves and flat-ribbed underarm panels merge out of the way of straps to eliminate chafe. made of 4-oz 100% polyester 54% recycled with gladiodor natural odor control for the garment. recyclable through the common threads recycling program."
486,487,"cap 1 crew - when it comes to moisture, this long-sleeved tee has serious control issues. quick-wicking, fast-drying and super breathable, it soothes overcooked skin and also layers well in cooler conditions. the crew feels silky, and features an athletic cut, a rib-knit collar for softness and raglan sleeves that reduce chafe under packs. made from 3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment, it comes in a tuckable length and provides 15-upf sun protection. recyclable through the common threads recycling program."


Cluster 1 : 55 products


,id,text
115,116,"rockpile t-shirt - beauty and bouldering are stacked together with the rockpile t-shirt, featuring original artwork by elissa pfost. printed on 100% ringspun organic cotton using pvc- and phthalate-free inks, the shirt has taped shoulder seams that won t chafe under a pack. recyclable through the common threads recycling program."
321,322,"tarpon t-shirt - channel the determination of the silver king wearing the tarpon t-shirt, which features the work of florida keys-based artist tim borski. the shirt s made from durable 100% organic cotton and screen-printed using pvc- and phthalate-free inks for a tee that s easier on the environment. taped shoulder seams and ringspun fabric yarns are soft next-to-skin. recyclable through the common threads recycling program."
123,124,"squid t-shirt - artist tim tompkinson s original work renders the squid t-shirt into a coat of arms. printed on ringspun organic cotton 100% , the lightweight shirt is printed with pvc- and phthalate-free inks that are less harmful to the environment. taped shoulder seams provide a no-chafe fit. recyclable through the common threads recycling program."


Cluster 2 : 6 products


,id,text
30,31,"crosstown - crosstown or cross-country, this fully featured commuter backpack ensures that your life stays organized whether you re office-bound, hiking the steep and narrow, or somewhere in the global commute. inside its zippered main compartment, it features a suspended, self-adjusting padded laptop sleeve that handles a 15 to 17 laptop. a secondary zippered compartment opens to a portable office organizer system. with a front catch-all sleeve with hidden zipper that provides easy access to documents a stretch-woven water-bottle pocket and a fleece-lined top pocket that protects sunglasses or electronics 3-d spacer-mesh backpack straps and back padding for comfort and breathability an adjustable sternum strap with built-in safety whistle. made from 150-denier all-recycled polyester double weave with a polyurethane coating and a dwr durable water repellent finish."
463,464,"daily grind - this fully padded shoulder bag keeps you moving unchallenged through the daily grind. with a minimalist design, it protects a 17 or smaller laptop and helps you stay organized. the main pocket has a stretch-woven internal sleeve and is accessed by a simple flap with hook-and-loop closure. two organizer pockets one with a zippered security pocket keep smaller gear at your fingertips, and an easy-access document sleeve holds airline tickets, reading material or a notebook. carry it with the long, adjustable shoulder strap or top-mounted handle. made from 150-denier all-recycled polyester double weave with a polyurethane coating and a dwr durable water repellent finish."
435,436,"lightwire - sometimes you need to leave the office in the rearview mirror. the lightwire has plenty of room for a jacket and lunch in addition to its fully padded computer sleeve, which protects a 15 to 17 pc laptop. the dedicated flat storage section holds files and magazines, and an external pocket offers one-handed access to an office organizer. highly breathable 3-d spacer-mesh shoulder straps and padded back panel wick away moisture. other features include an mp3 pocket, sternum strap with built-in safety whistle, a tuck-away waistbelt, stretch water-bottle pocket and an easy-access external security pocket. made from 150-denier all-recycled polyester double weave with a polyurethane coating and a dwr durable water repellent finish."


... (14 outputs left out)

Cluster 10 : 15 products


,id,text
381,382,"rain shadow jkt - staying inside isn t the only way to stay dry - enjoy the trails in all conditions with fully seam-sealed waterproof/breathable protection. the rain shadow is made from a packable, lightweight and tear-resistant 2.5-layer 100% nylon fabric with a waterproof/breathable h2no barrier and a deluge dwr durable water repellent finish. on the inside, a raised pattern diffuses and channels moisture away from skin, aids in hassle-free layering, and enhances long-term durability. updated with microfleece lining on the neck and chin, waterproof handwarmer pockets no longer mesh-lined with zipper garages, and external zippers that are now water-resistant without storm flaps . also features a roll-down, 2-way-adjustable, helmet-compatible hood, a 2-way center-front zipper, self-fabric cuff tabs and a drawcord hem."
211,212,"rain shadow jkt - staying inside isn t the only way to stay dry - enjoy the trails in all conditions with fully seam-sealed waterproof/breathable protection. the rain shadow is made from a packable, lightweight and tear-resistant 2.5-layer 100% nylon fabric with a waterproof/breathable h2no barrier and a deluge dwr durable water repellent finish. on the inside, a raised pattern diffuses and channels moisture away from skin, aids in hassle-free layering, and enhances long-term durability. updated with microfleece lining on the neck and chin, waterproof handwarmer pockets no longer mesh-lined with zipper garages, and external zippers that are now water-resistant without storm flaps . also features a roll-down, 2-way-adjustable, helmet-compatible hood, a 2-way center-front zipper, self-fabric cuff tabs and a drawcord hem."
326,327,"torrentshell jkt - tropical rain forests share at least one thing with the patagonia ice cap wet weather. for trekking and hiking in drenching conditions, our new torrentshell provides 2.5-layer waterproof/breathable protection that s durable, lightweight and without frills. the jacket stuffs into one of two zippered handwarmer pockets and has a clip-in loop the hood fits snugly over a helmet, rolls down, stows away, and has a laminated visor and the center zipper has exterior and interior storm flaps. with pit zips for added ventilation, a microfleece-lined neck and chin and a drawcord hem."


Cluster 11 : 6 products


,id,text
254,255,"solid sunamee btm - stay-put sunamees fit straight across the hips, with full coverage in back. lined front. made from a formfitting blend of 82% nylon/18% spandex."
261,262,"print sunamee btm - stay-put sunamees fit straight across the hips, with full coverage in back. lined front. made from a formfitting blend of 82% nylon/18% spandex."
260,261,"print paries btm - styled like a boy short, our paries bottoms fit low and straight across the hips, with medium coverage in front and full coverage in back. lined front. made from our improved fabric blend of 82% nylon/18% spandex."


# 